# checking ARI

Load Ramon's relation model and figure out: how many classes, which way the arrows point, how slow it is.

### setup

In [ ]:
import os, sys, subprocess
REPO = "https://github.com/ookino/rlvr-argument-mining.git"
NAME = "rlvr-argument-mining"
if os.path.basename(os.getcwd()) != NAME:
    if not os.path.isdir(NAME):
        subprocess.run(["git", "clone", REPO], check=True)
    os.chdir(NAME)
subprocess.run(["git", "fetch", "-q"], check=False)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=False)
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "networkx", "pyyaml"], check=True)
print("ok")

### load the model (downloads first time)

In [ ]:
from reward.ari import ARI
ari = ARI()
print("on:", ari.device)

### how many relation classes?

In [ ]:
print(ari.id2label)

### which way do the arrows point? premises 0,1 should point into conclusion (2)

In [ ]:
from reward.xaif_build import build_trace
text = ("All ravens are black.\n"
        "Every bird in this garden is a raven.\n"
        "Therefore every bird in this garden is black.\n"
        "The answer is black.")
trace = build_trace(text)
for i, s in enumerate(trace.steps):
    print(i, s)
res = ari.identify(trace.steps, window=None)
for r in res.relations:
    print(f"{r.source} --{r.kind}--> {r.target} ({r.confidence:.2f})")

### how slow? all-pairs vs neighbours

In [ ]:
import time
from reward.ari import ARI
steps = trace.steps * 6   # ~24 steps
for w in (None, 2):
    t = time.time()
    n = ari.identify(steps, window=w).n_pairs_scored
    print(f"window={w}: {n} pairs in {time.time()-t:.2f}s")